# Anima 生图 - Colab 免费 T4 版

跑完第 3 格会打印一个 `https://xxx.trycloudflare.com` 公网地址，用本机脚本 POST 即可取图。

- 模型：`circlestone-labs/Anima-Base-v1.0-Diffusers`（官方，6.8GB，T4 16GB 装得下）
- 单张预计 5-10 分钟
- 会话上限 12 小时（Colab 免费档硬限）
- 无 NSFW 审查


In [ ]:
#@title 1. 安装依赖
!pip -q install diffusers transformers accelerate safetensors gradio flask flask-cors
import subprocess, sys
r = subprocess.run([sys.executable, '-c', 'import diffusers; print(diffusers.__version__)'], capture_output=True, text=True)
print('diffusers', r.stdout.strip() or r.stderr[-300:])
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


In [ ]:
#@title 2. 加载模型
import torch, os
from diffusers import ModularPipeline

MODEL_ID = 'circlestone-labs/Anima-Base-v1.0-Diffusers'

pipe = ModularPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16)
pipe.load_components()
pipe.to('cuda')
pipe.set_progress_bar_config(disable=True)
print('[+] model loaded, VRAM:', torch.cuda.get_device_properties(0).total_memory / 1024**3, 'GB')


In [ ]:
#@title 3. 启动 HTTP 服务 + cloudflared 公网穿透
import threading, uuid, os, io, json, base64, subprocess, time
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS

app = Flask(__name__)
CORS(app)
JOBS = {}

@app.route('/health', methods=['GET'])
def health():
    return jsonify(ok=True, queue=len(JOBS))

@app.route('/generate', methods=['POST'])
def generate():
    d = request.get_json(force=True)
    prompt = (d.get('prompt') or '').strip()
    if not prompt:
        return jsonify(error='prompt required'), 400
    jid = uuid.uuid4().hex
    JOBS[jid] = {'status': 'queued', 'created': time.time()}
    t = threading.Thread(target=work, args=(jid, d), daemon=True)
    t.start()
    return jsonify(id=jid)

@app.route('/status/<jid>', methods=['GET'])
def status(jid):
    j = JOBS.get(jid)
    if not j:
        return jsonify(error='unknown id'), 404
    return jsonify({k: v for k, v in j.items() if k != 'png'})

@app.route('/image/<jid>', methods=['GET'])
def image(jid):
    j = JOBS.get(jid)
    if not j or j.get('status') != 'done' or 'png' not in j:
        return jsonify(error='not ready'), 404
    return send_file(io.BytesIO(j['png']), mimetype='image/png')

def work(jid, d):
    try:
        JOBS[jid]['status'] = 'running'
        # Anima reads Danbooru tags + natural language; this pair is the
        # recommended quality wrapper.
        p = d['prompt']
        if not p.startswith('masterpiece'):
            p = 'masterpiece, best quality, score_7, safe, ' + p
        neg = d.get('negative') or 'worst quality, low quality, blurry, jpeg artifacts'
        w = int(d.get('width') or 832)
        h = int(d.get('height') or 1216)
        steps = int(d.get('steps') or 25)
        seed = int(d.get('seed') or 0)
        t0 = time.time()
        out = pipe(prompt=p, negative_prompt=neg, height=h, width=w,
                   num_inference_steps=steps, generator=torch.Generator('cuda').manual_seed(seed))
        img = out.images[0]
        buf = io.BytesIO()
        img.save(buf, format='PNG')
        JOBS[jid].update(status='done', seconds=round(time.time() - t0, 1), png=buf.getvalue())
    except Exception as e:
        JOBS[jid].update(status='error', error=f'{type(e).__name__}: {e}'[:400])

threading.Thread(target=lambda: app.run(host='0.0.0.0', port=7860, debug=False, use_reloader=False), daemon=True).start()

# cloudflared quick tunnel: no Cloudflare account needed, prints a trycloudflare.com URL
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
proc = subprocess.Popen(['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860', '--no-autoupdate'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in proc.stdout:
    if 'trycloudflare.com' in line:
        import re
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            url = m.group(0); break
print('=' * 60)
print('PUBLIC URL:', url)
print('=' * 60)
print('本机用法: colab_gen.sh "prompt" <url>')


In [ ]:
#@title 4. 本地测试：让 Colab 自己出一张
import requests, time
r = requests.post(url + '/generate', json={
    'prompt': '1girl, solo, blue long hair, sitting on bed, knees spread apart, both feet planted on bedsheet, looking at viewer, white t-shirt, dark blue shorts, soft morning window light, volumetric lighting, detailed skin texture',
    'width': 832, 'height': 1216, 'steps': 25, 'seed': 12345,
})
jid = r.json()['id']
print('job', jid)
for _ in range(60):
    s = requests.get(url + '/status/' + jid).json()
    print(s.get('status'), s.get('seconds', ''), s.get('error', ''))
    if s.get('status') in ('done', 'error'):
        break
    time.sleep(10)
if requests.get(url + '/status/' + jid).json().get('status') == 'done':
    img = requests.get(url + '/image/' + jid).content
    from PIL import Image
    import io
    im = Image.open(io.BytesIO(img))
    print('size:', im.size, 'bytes:', len(img))
    display(im)


## 注意

- **会话 12 小时上限**是 Colab 免费档硬限，到点进程会被杀，需重跑前 3 格。
- 每张图会重新走 25 步 diffusion；想加速把 `steps` 降到 15-18，质量损失不大。
- `ModularPipeline` 的 `load_components()` 可重复调用换 LoRA，但 LoRA 需自行下载到 `pipe.load_lora_weights()`。
- cloudflared quick tunnel 地址每次重启都变，重新跑第 3 格即可。
